<a href="https://colab.research.google.com/github/Muqqadas30/fsdl-my-labs/blob/main/lab07/lab07_deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gradio --quiet

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets
from PIL import Image
import numpy as np

device = torch.device("cpu")

In [2]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

train_raw = datasets.FashionMNIST(root="./data", train=True, download=True)
x_train = train_raw.data.float().unsqueeze(1) / 255.0
y_train = train_raw.targets

model = SimpleCNN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(3):
    model.train()
    for i in range(0, len(x_train), 64):
        xb = x_train[i:i+64]
        yb = y_train[i:i+64]
        optimizer.zero_grad()
        loss = F.cross_entropy(model(xb), yb)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} done, loss: {loss.item():.4f}")

model.eval()

100%|██████████| 26.4M/26.4M [00:01<00:00, 18.6MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 313kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.66MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 8.81MB/s]


Epoch 1 done, loss: 0.2777
Epoch 2 done, loss: 0.2122
Epoch 3 done, loss: 0.1774


SimpleCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1568, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

In [3]:
example_input = torch.randn(1, 1, 28, 28)

scripted_model = torch.jit.trace(model, example_input)

scripted_model.save("fashion_mnist_scripted.pt")
print("Model saved as TorchScript!")

loaded_model = torch.jit.load("fashion_mnist_scripted.pt")
print("Model loaded successfully, ready for inference")

Model saved as TorchScript!
Model loaded successfully, ready for inference


In [4]:
class FashionMNISTPredictor:
    def __init__(self, model_path="fashion_mnist_scripted.pt"):
        self.model = torch.jit.load(model_path)
        self.model.eval()
        self.class_names = class_names

    def predict(self, image):
        if isinstance(image, np.ndarray):
            image = Image.fromarray(image).convert("L").resize((28, 28))

        img_array = np.array(image).astype(np.float32) / 255.0
        img_tensor = torch.tensor(img_array).unsqueeze(0).unsqueeze(0)

        with torch.no_grad():
            output = self.model(img_tensor)
            probs = F.softmax(output, dim=1)
            pred_idx = torch.argmax(probs, dim=1).item()
            confidence = probs[0][pred_idx].item()

        return f"{self.class_names[pred_idx]} ({confidence:.1%} confident)"


predictor = FashionMNISTPredictor()

sample_img = Image.fromarray((x_train[0, 0].numpy() * 255).astype(np.uint8))
print(predictor.predict(sample_img))

Ankle boot (99.2% confident)


In [5]:
import gradio as gr

def gradio_predict(image):
    if image is None:
        return "Koi image upload nahi hui"
    return predictor.predict(image)


frontend = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Image(type="numpy", image_mode="L"),
    outputs=gr.Textbox(label="Prediction"),
    title="Fashion-MNIST Classifier",
    description="Kisi bhi clothing item ki image upload karo, model uski category batayega."
)

frontend.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c4b7875e943782f9fa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
frontend.close()

Closing server running on port: 7860
